# B3 — Training and optimization improvements

B3.1 (optimizer/learning-rate screening) is preserved as a negative result. This revised notebook runs B3.2 scheduler and early-stopping experiments, then B3.3 validation-only threshold calibration. Frozen test commands are created only after a predeclared validation benefit passes. Existing B2/B3 artifacts are reused from Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
OUTPUT_DIR = Path('/content/drive/MyDrive/ADVLSI2_B3/b3_optimization')
B2_CHECKPOINTS = Path('/content/drive/MyDrive/ADVLSI2_B2/b2_baselines/checkpoints')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert B2_CHECKPOINTS.is_dir(), f'Missing B2 checkpoints: {B2_CHECKPOINTS}'
print(f'Persistent results: {OUTPUT_DIR}')

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

REPOSITORY = 'https://github.com/nocleo/ADVLSI2_Project_updated.git'
BRANCH = 'agent/b3-training-optimization'
CHECKOUT = Path(tempfile.mkdtemp(prefix='advlsi-b3-')) / 'ADVLSI2_Project_updated'

subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPOSITORY, str(CHECKOUT)],
    check=True,
)
os.chdir(CHECKOUT)

import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before running B3.'
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
command = [
    sys.executable,
    'scripts/run_b3_extension.py',
    '--python', sys.executable,
    '--output-dir', str(OUTPUT_DIR),
    '--b2-checkpoints', str(B2_CHECKPOINTS),
]
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
from IPython.display import Markdown, display
report = OUTPUT_DIR / 'EXTENSION_README.md'
if report.exists():
    display(Markdown(report.read_text()))
else:
    print('No final report was written; inspect the preceding error and rerun to resume.')

In [ ]:
from google.colab import files
archive_base = '/content/ADVLSI2_B3_results'
archive = shutil.make_archive(archive_base, 'zip', OUTPUT_DIR)
files.download(archive)